In [512]:
import numpy as np
import pandas as pd
import geopandas as gpd
from matplotlib import pyplot as plt

# NWT Databases

## OROGO

1) NTGS
2) OROGO

In [513]:
# OROGO database
OROGO = pd.read_csv('source_data/wells/NWT/orogo-well-status-updated-2026-04-10.csv').drop(columns=['NAD 27 Lat','NAD 27 Long','Unnamed: 16','Unnamed: 17']).dropna(how='all')
OROGO = gpd.GeoDataFrame(data=OROGO,geometry=gpd.points_from_xy(x=OROGO['NAD_83_LongDD'],y=OROGO['NAD_83_LatDD']),crs='NAD1983')
OROGO['Name'] = OROGO['Well Name'].str.upper()
OROGO = OROGO.set_index('Name',drop=False)
OROGO = OROGO.drop(columns='geometry')
print(len(OROGO))
print(OROGO.columns)

685
Index(['Well ID', 'Well Name', 'Last Operator', 'Current or Last Owner',
       'Well Status', 'Classification', 'First SPUD year',
       'Latest SPUD or Start Date', 'Latest Rig Release or End Date',
       'Land Title', 'Region', 'NAD_83_LatDD', 'NAD_83_LongDD', 'UWI', 'Name'],
      dtype='str')


## NTGS Open Report 2009-03

In [ ]:
NTGS_2009_03 = [r'source_data\wells\NWT\2009-03_Well_Sites\Well_Sites.shp',r'source_data\wells\NWT\2009-03_Well_Sites\Well_Sites_Peripheral.shp']
NTGS_2009_03 = pd.concat([gpd.read_file(fn).set_index('UWI') for fn in NTGS_2009_03]).to_crs('NAD1983')
NTGS_2009_03['Name'] = NTGS_2009_03['WELL_NAME'].str.upper()
NTGS_2009_03['NAD_83_LatDD'] = NTGS_2009_03.geometry.y
NTGS_2009_03['NAD_83_LongDD'] = NTGS_2009_03.geometry.x
NTGS_2009_03 = NTGS_2009_03.reset_index().set_index('Name',drop=False)
NTGS_2009_03 = NTGS_2009_03.drop(columns='geometry')
print(len(NTGS_2009_03))
print(NTGS_2009_03.columns)

549
Index(['UWI', 'WID', 'WELL_NAME', 'OPERATOR', 'STATUS', 'CLASSIFICA',
       'SPUD_DATE', 'RIG_RELEAS', 'LAT', 'LONG', 'DATUM', 'NORTHING',
       'EASTING', 'ZONE', 'SOURCE', 'REFERENCE', 'Name', 'NAD_83_LatDD',
       'NAD_83_LongDD'],
      dtype='str')


## NTGS Open Report 2019-015

In [515]:
# Report 2019-015
NTGS_2019_015_g = gpd.read_file(r'source_data\wells\NWT\2019-015_Shapefiles\Gas_Resource.shp').set_index('Well_Names',drop=False).rename(columns={'Recoverabl':'Recoverabl_gas'})
NTGS_2019_015_o = gpd.read_file(r'source_data\wells\NWT\2019-015_Shapefiles\Oil_Resource.shp').set_index('Well_Names',drop=False).rename(columns={'Recoverabl':'Recoverabl_oil'})



NTGS_2019_015 = pd.concat([
    NTGS_2019_015_g,
    NTGS_2019_015_o.loc[~NTGS_2019_015_o.index.isin(NTGS_2019_015_g.index),NTGS_2019_015_o.columns[NTGS_2019_015_o.columns!='Recoverabl_oil']]
]).to_crs('NAD1983')
NTGS_2019_015 = NTGS_2019_015.join(NTGS_2019_015_o[['Recoverabl_oil']],how='left')
NTGS_2019_015['NAD_83_LatDD'] = NTGS_2019_015.geometry.y
NTGS_2019_015['NAD_83_LongDD'] = NTGS_2019_015.geometry.x
NTGS_2019_015 = NTGS_2019_015.drop(columns='geometry')
NTGS_2019_015['Name'] = NTGS_2019_015['Well_Names'].str.upper()


print(len(NTGS_2019_015))
print(NTGS_2019_015.columns)

173
Index(['NEB_Study', 'NEB_Field', 'WID', 'UWI', 'Well_Names', 'Discovery',
       'Field_Code', 'Pool_Codes', 'Pool_Seq', 'Fluid_Type', 'Initial_Ma',
       'Recoverabl_gas', 'Bot_Hole_L', 'Bot_Hole_1', 'Recoverabl_oil',
       'NAD_83_LatDD', 'NAD_83_LongDD', 'Name'],
      dtype='str')


# Yukon Data

In [516]:
GeoYukon = gpd.read_file(r'source_data\wells\yukon\Oil_and_Gas_Wells_50k.shp').to_crs('NAD1983').rename(columns={'WELL_UWI':'UWI'})
GeoYukon['Name'] = GeoYukon['WELL_NAME'].str.upper()
GeoYukon = GeoYukon.set_index('Name',drop=False)
GeoYukon['NAD_83_LatDD'] = GeoYukon.geometry.y
GeoYukon['NAD_83_LongDD'] = GeoYukon.geometry.x
GeoYukon = GeoYukon.drop(columns='geometry')
print(len(GeoYukon))
print(GeoYukon.columns)


76
Index(['WELL_NAME', 'WELL_LABEL', 'LIC_NUM', 'UWI', 'LICENSEE', 'CLASS',
       'TYPE', 'STATUS', 'DATE', 'OPERATOR', 'PROD_DATE', 'ABANDON',
       'LOCATION', 'BASIN_NAME', 'LAT_DD', 'LONG_DD', 'Name', 'NAD_83_LatDD',
       'NAD_83_LongDD'],
      dtype='str')


# Federal Data

## Basin

In [583]:
Basin = pd.read_csv('source_data/wells/Basin/BASIN_well_coords.txt',delimiter='\t',skiprows=3).drop(columns=['Latitude (NAD27)','Longitude (NAD27)','Northing (NAD27)','Easting (NAD27)'])
Basin2 = pd.read_csv('source_data/wells/Basin/q1787332704.txt',delimiter='\t',skiprows=3).rename(columns={'Unique Well Identifier':'UWI'})
Basin2['UWI'] = Basin2['UWI'].str.replace(' ','')
Basin = pd.merge(Basin,Basin2[['Well Name','GSC #','Original Spud Year','Operator','Status','UWI']],on='Well Name',how='left')#.set_index('UWI')
Basin['Name'] = Basin['Well Name'].str.upper()
Basin = Basin.set_index('Name',drop=False)
Basin = Basin.sort_values(by='Original Spud Year')
Basin = Basin.loc[~Basin[['Name','UWI']].duplicated(keep='first')].copy()
Basin = Basin.loc[Basin['Latitude (NAD83)']>60]
print(len(Basin))
print(Basin.columns)

179
Index(['Well Name', 'Basin', 'Subbasin', 'Latitude (NAD83)',
       'Longitude (NAD83)', 'Northing (NAD83)', 'Easting (NAD83)',
       'Zone (UTM)', 'GSC #', 'Original Spud Year', 'Operator', 'Status',
       'UWI', 'Name'],
      dtype='str')


## CER

* Missing UWI

In [518]:
CER = pd.read_csv(r'source_data\wells\CER\Frontier_Wells_Inuvik_Region.csv')
CER['Name'] = CER['WELL_NAME'].str.upper()
CER['UWI'] = None
CER = CER.set_index('Name',drop=False)
print(len(CER))
print(CER.columns)

367
Index(['WELL_ID', 'WELL_NAME', 'OPERATOR', 'STATUS', 'LAND_TITLE', 'REGION',
       'NAD_83_LAT', 'NAD_83_LON', 'Name', 'UWI'],
      dtype='str')


## GSC Open Files 

* some missing key identifiers, so pre-synthesis needed for better record matching

## GSC Open File 6959

In [519]:
GSC_6959 = pd.read_csv(r'source_data\wells\GSC\GSC_OF_6959.csv')#@.set_index('UWI')
GSC_6959['Name'] = GSC_6959['Well Short Name'].str.upper()
GSC_6959=GSC_6959.set_index('Name',drop=False)
print(len(GSC_6959))
print(GSC_6959.columns)

265
Index(['UWI', 'Well Short Name', 'EASTING', 'NORTHING', 'MAP_DATUM',
       'SURF_LAT', 'SURF_LONG', 'UTMZONE', 'Well Status', 'KB (m)', 'GL (m)',
       'Final Interpretation, Ice-bearing permafrost base, Base of fully frozen (mKB)',
       'Final Interpretation Ice-bearing permafrost base Base of fully frozen (mGL/SF)',
       'Final Interpretation Ice-bearing permafrost base Quality', 'Formation',
       'Final Interpretation Ice-bearing permafrost base Base of partially frozen (mKB)',
       'Final Interpretation Ice-bearing permafrost base Base of partially frozen (mGL/SF)',
       'Final Interpretation Ice-bearing permafrost base Quality.1',
       'Final Interpretation Ice-bearing permafrost base Transition zone thickness (m)',
       'Final Interpretation Ice-bearing permafrost base Formation/ Sequence ',
       'Final Interpretation Ice-bearing permafrost base PF present',
       'Final Interpretation Ice-bearing permafrost base Data used for final pick*',
       'Base of 

## GSC Open File 4828 & 327948

In [561]:
# Missing UWI
GSC_4828 = pd.read_csv(r'source_data\wells\GSC\GSC_OF_4828.csv')
for c in ['Latitude', 'Longitude']:
    DMS = pd.DataFrame(GSC_4828[c].replace(r'[^0-9.]',' ',regex=True).str.split().to_list(),columns=['D','M','S'],index=GSC_4828['Well Name'])
    GSC_4828[c] = (DMS['D'].astype('float')+DMS['M'].astype('float')/60+DMS['S'].astype('float')/3600).values
GSC_4828['Name'] = GSC_4828['Well Name'].str.upper()
GSC_4828 = GSC_4828.set_index('Name',drop=False)

# # Missing coordinates but contains UWI
GSC_327948 = pd.read_csv(r'source_data\wells\GSC\GSC_OF_327948.csv')
GSC_327948['Name'] = GSC_327948['Well name'].str.upper()
GSC_327948 = GSC_327948.set_index('Name',drop=False)
GSC_327948 = GSC_327948.loc[GSC_327948['UWI'].duplicated(keep='first')].copy()
# Get from GSC_6959 & 327948
GSC_4828=GSC_4828.join(GSC_6959[['UWI']],how='left')
for k,v in GSC_4828.loc[GSC_4828['UWI'].isna(),'Name'].isin(GSC_327948['Name']).items():
    if v:
        GSC_4828.loc[GSC_4828.index==i,'UWI'] = GSC_327948.loc[GSC_327948.index==i,'UWI']
print(len(GSC_4828))
print(GSC_4828.columns)

print(len(GSC_327948))
print(GSC_327948.columns)

263
Index(['Well No.', 'Company', 'Well Name', 'Latitude', 'Longitude', 'KBm',
       'GLm', 'TD(ft)', 'TD(m)', 'TVD', 'Name', 'UWI'],
      dtype='str')
40
Index(['UWI', 'Well name', 'Top of overpressure zone mSl',
       'Top of overpressure zone mGl', 'Top of overpressure zone quality',
       'Sequence', 'Comments', 'Name'],
      dtype='str')


# Consultant Reports

## Arktis report for ILA

In [521]:
ILA = pd.read_csv(r'source_data\wells\ISR\ISR_well_report_B1.csv')
ILA = ILA.loc[~ILA['Latitude (NAD83) -Well Post'].isna()].copy()
for c in ['Latitude (NAD83) -Well Post','Longitude (NAD83) -Well Post']:
    DMS = pd.DataFrame(ILA[c].replace(r'[^0-9.]',' ',regex=True).str.split().to_list(),columns=['D','M','S'],index=ILA['Well Name'])
    ILA[c] = (DMS['D'].astype('float')+DMS['M'].astype('float')/60+DMS['S'].astype('float')/3600).values
ILA['Name'] = ILA['Well Name'].str.upper()
ILA = ILA.set_index('Name',drop=False)

print(len(ILA))
print(ILA.columns)

227
Index(['WID', 'Consortium', 'Current Owner', 'Land Owner', 'Well Name', 'UWI',
       'Class', 'Status', 'Latitude (NAD83) -Well Post',
       'Longitude (NAD83) -Well Post', 'Region', 'Original Spud Date',
       'Original Rig Release Date', 'Depth (m)', 'Notes', 'Name'],
      dtype='str')


## LTLC Report

In [522]:
LTLC = pd.read_csv('source_data/wells/misc/LTLC_report.csv')
LTLC = LTLC.set_index('Name',drop=False)
LTLC['UWI'] = None


print(len(LTLC))
print(LTLC.columns)

92
Index(['Name', 'Operator', 'Spud_Date', 'RIG RELEASE', 'DRILLING PLATFORM',
       'WATER DEPTH (M)', 'UWI'],
      dtype='str')


# Aggregating

In [ ]:
ds = {
'OROGO':OROGO,
'NTGS_2009_03':NTGS_2009_03,
'NTGS_2019_015':NTGS_2019_015,
'GeoYukon':GeoYukon,
'Basin':Basin,
'CER':CER,
'GSC_327948':GSC_327948,
'GSC_4828':GSC_4828,
'GSC_6959':GSC_6959,
'ILA':ILA,
'LTLC':LTLC
}
for key,value in ds.items():
	ds[key]['Data_Source'] = key
	ds[key]['UWI'] = ds[key]['UWI'].fillna(None)
	# Some UWI match between datasets except they have 1 or 2 at end instead of 0, this could be a typo, or a record of "events", b
	ds[key]['UWI'] = ds[key]['UWI'].str[0:15]+'0'
	# Fix two inconsistencies
	if key == 'NTGS_2009_03':
		ds[key]['UWI'] = ds[key]['UWI'].replace({'300F296830134300':'300F297000134000'})
	elif key == 'GSC_327948':
		ds[key]['UWI'] = ds[key]['UWI'].replace({'302I457030133302':'302P357020134000'})
	ds[key]['Name'] = ds[key]['Name'].str.replace('YA-YA','YA YA').str.replace('YAYA','YA YA')
	ds[key].index = ds[key].index.str.replace('YA-YA','YA YA').str.replace('YAYA','YA YA')
	
AllSources = pd.concat([d[['Name','UWI','Data_Source']] for d in ds.values()]).sort_index()
AllSources['UWI'] = AllSources['UWI'].fillna(None)
for i,row in AllSources.groupby(AllSources.index).agg(list).iterrows():
	if len(row['Data_Source']) == 1:
		sgl.append(i)
	elif len(row['Data_Source']) > 1:
		UWI_set = set(row['UWI'])
		if None in UWI_set and len(UWI_set)==2:
			UWI = ''.join([f"{u}" for u in UWI_set if u is not None])
			sources = [row['Data_Source'][i] for i,v in enumerate(row['UWI']) if v is None]
			for s in sources:
				ds[s].loc[ds[s].index==i,'UWI']=UWI



AllSources = pd.concat([d[['Name','UWI','Data_Source']] for d in ds.values()]).sort_index()
AllSources.groupby(['Name']).count()['Data_Source'].sort_values()
AllSources.loc[AllSources['UWI'].isna()].groupby('Data_Source').count()

,Name,UWI
Data_Source,,
Basin,16,0
CER,14,0
GSC_4828,4,0
LTLC,7,0
